# XAUUSD M5 - Data Import & EDA

**Data source:** Dukascopy m5 OHLCV, originally fetched with the use of dukascopy-node app (credits to https://github.com/Leo4815162342) and my ETL pipeline (https://github.com/ady7ady7/cross-market-etl-pipeline) and then stored in DigitalOcean PostgreSQL

**Coverage:** 2021-01-03 - 2026-07-19 (1700+ days, 5+ years of data, 391k+ 5-minute candles in total)

**Note on volume:** Dukascopy volume is tick volume (quote count), not real traded size. Spot XAUUSD has no centralized exchange, so true volume is unavailable. Volume here is a proxy for quoting activity.

The first step is to establish connection with my database, which contains all market data fetched through Dukascopy-node app*. 

(NOT RELEVANT FOR THIS ANALYSIS)
*I've built a pipeline to import and save market data in any DB - please feel free to check the etl-cross-market-pipeline repo on my GitHub if you're interested.

In [15]:
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

load_dotenv()

engine = create_engine(
    f"postgresql+psycopg2://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT', 5432)}/{os.getenv('DB_NAME')}"
)

After establishing connection it's time to simply fetch market data.
I also do some basic SQL operations at this point to make my life easier later:

- get the time in the et_time (America Eastern Time), since it's the timezone that's the most relevant for Gold (but also for the most liquid financial instruments in most cases)
- extract the day of the week also in the ET
- I do the timestamp ordering, just in case

It's also the time to have a first look at my data.

In [16]:
query = """
SELECT
    timestamp AT TIME ZONE 'America/New_York' AS et_time,
    open, 
    high, 
    low, 
    close, 
    volume,
    EXTRACT(DOW FROM timestamp AT TIME ZONE 'America/New_York') AS day_of_week,
    (timestamp AT TIME ZONE 'America/New_York')::date AS trade_date
FROM xauusd_m5_tradfi_ohlcv
ORDER BY timestamp
"""

df = pd.read_sql(query, engine)
df.head()

,et_time,open,high,low,close,volume,day_of_week,trade_date
0,2021-01-03 17:00:00,1904.998,1910.898,1903.288,1908.850,0.68,0.0,2021-01-03
1,2021-01-03 17:05:00,1908.878,1909.258,1907.618,1908.568,0.46,0.0,2021-01-03
2,2021-01-03 17:10:00,1908.518,1909.405,1907.665,1908.805,0.21,0.0,2021-01-03
3,2021-01-03 17:15:00,1908.785,1909.678,1907.978,1909.358,0.09,0.0,2021-01-03
4,2021-01-03 17:20:00,1909.368,1910.588,1909.005,1909.048,0.14,0.0,2021-01-03


In [17]:
print(f"Shape: {df.shape}")
df['day_of_week'] = df['day_of_week'].astype(int)
df['trade_date'] = pd.to_datetime(df['trade_date'])


print(df.dtypes)

print(f"\nDate range: {df['et_time'].min()} - {df['et_time'].max()}")
print(f"Unique trade dates: {df['trade_date'].nunique()}")

print(f"Null counts: {df.isnull().sum()}")


Shape: (391955, 8)
et_time        datetime64[us]
open                  float64
high                  float64
low                   float64
close                 float64
volume                float64
day_of_week             int64
trade_date      datetime64[s]
dtype: object

Date range: 2021-01-03 17:00:00 - 2026-07-19 17:55:00
Unique trade dates: 1723
Null counts: et_time        0
open           0
high           0
low            0
close          0
volume         0
day_of_week    0
trade_date     0
dtype: int64


Now this is somewhat expected, as market data was initially gathered through Dukascopy, which is one of the most prominent financial brokers - it's licensed and well known on the market.

However, we also have to keep in mind that volume data they offer is not real market data, but only their platform(s) data, and it's very important. Basically it means that volume analysis is not the right choice in this case, as it's not tied to the real market volume.

There aren't any nulls, but perhaps we're missing some days or candles - maybe they're simply not there - they wouldn't appear as NULL.

In [18]:
expected_bars = pd.date_range(
    start=df['et_time'].min(),
    end=df['et_time'].max(),
    freq='5min'
)
missing_bars = expected_bars.difference(df['et_time'])
print(f"Expected 5-min bars: {len(expected_bars)}")
print(f"Actual bars: {len(df)}")
print(f"Missing bars: {len(missing_bars)}")

Expected 5-min bars: 582636
Actual bars: 391955
Missing bars: 190681


As we can see, about 32.7% of our candles is missing.

However, we must also be aware of the fact that financial markets run only from late Sunday until Friday, and that there are also holidays, when the markets are closed. With that in mind we can move on to do some more robust check, and look which days have the most missing bars.


In [19]:
missing_df = pd.DataFrame({'et_time': missing_bars})
missing_df['day_of_week'] = missing_df['et_time'].dt.dayofweek  # 0=Mon, 6=Sun

print("Missing bars by day of week:")
print(missing_df['day_of_week'].value_counts().sort_index())

weekday_missing = missing_df[missing_df['day_of_week'] < 5].copy()
weekday_missing['hour'] = weekday_missing['et_time'].dt.hour

print("Missing weekday bars by hour:")
print(weekday_missing['hour'].value_counts().sort_index())

Missing bars by day of week:
day_of_week
0     5448
1     3911
2     4173
3     5022
4    31564
5    83232
6    57331
Name: count, dtype: int64
Missing weekday bars by hour:
hour
0       228
1       228
2       229
3       228
4       228
5       228
6       228
7       228
8       228
9       229
10      228
11      303
12      486
13      696
14      780
15    10572
16     9302
17     3651
18     3638
19     3636
20     3636
21     3636
22     3636
23     3636
Name: count, dtype: int64


To run a proper check, I will actually install a new library - pandas_market_calendars, which contains the whole trading calendar for every symbol




In [ ]:
import pandas_market_calendars as mcal
cal = mcal.get_calendar('CMEGlobex_Gold')

start_date = df['et_time'].min()
end_date = df['et_time'].max()
calendar = cal.schedule(start_date = start_date, end_date = end_date)
print(calendar)



                         market_open              market_close
2021-01-04 2021-01-03 23:00:00+00:00 2021-01-04 22:00:00+00:00
2021-01-05 2021-01-04 23:00:00+00:00 2021-01-05 22:00:00+00:00
2021-01-06 2021-01-05 23:00:00+00:00 2021-01-06 22:00:00+00:00
2021-01-07 2021-01-06 23:00:00+00:00 2021-01-07 22:00:00+00:00
2021-01-08 2021-01-07 23:00:00+00:00 2021-01-08 22:00:00+00:00
...                              ...                       ...
2026-07-13 2026-07-12 22:00:00+00:00 2026-07-13 21:00:00+00:00
2026-07-14 2026-07-13 22:00:00+00:00 2026-07-14 21:00:00+00:00
2026-07-15 2026-07-14 22:00:00+00:00 2026-07-15 21:00:00+00:00
2026-07-16 2026-07-15 22:00:00+00:00 2026-07-16 21:00:00+00:00
2026-07-17 2026-07-16 22:00:00+00:00 2026-07-17 21:00:00+00:00

[1430 rows x 2 columns]


Now we have a proper list of trading days without all the holidays and weekends, and we can properly calculate the expected bars and then check if there are any missing ones in a proper way.

Gold trades Sun 18:00 ET through Fri 17:00 ET. This means Sunday bars appear in the dataset as a separate `trade_date`, but they logically belong to the Monday session. We drop Sunday bars to align with the CME calendar (Mon–Fri trading days only).


In [26]:
type(calendar)
#it's a pandas DF

df = df[df['et_time'].dt.dayofweek != 6].copy()
print(f"Rows after dropping Sundays: {len(df)}")
print(f"Unique trade_dates: {df['trade_date'].nunique()}")



Rows after dropping Sundays: 366042
Unique trade_dates: 1437


So now we are sitting at 1430 days in our calendar vs 1437 days in the dataframe - it's close enough and if the dates aligh we shouldn't really bother with that much more.

In [27]:
calendar_et = calendar.copy()
calendar_et['market_open'] = calendar['market_open'].dt.tz_convert('America/New_York').dt.tz_localize(None)
calendar_et['market_close'] = calendar['market_close'].dt.tz_convert('America/New_York').dt.tz_localize(None)

df_dates = set(df['trade_date'].dt.date)
cal_dates = set(calendar_et.index.date)

only_in_df = df_dates - cal_dates
only_in_cal = cal_dates - df_dates

print(f"Days in df but not in calendar: {len(only_in_df)}")
print(sorted(only_in_df))
print(f"\nDays in calendar but not in df: {len(only_in_cal)}")
print(sorted(only_in_cal))


Days in df but not in calendar: 8
[datetime.date(2022, 12, 26), datetime.date(2023, 1, 2), datetime.date(2023, 12, 25), datetime.date(2024, 1, 1), datetime.date(2024, 12, 25), datetime.date(2025, 1, 1), datetime.date(2025, 12, 25), datetime.date(2026, 1, 1)]

Days in calendar but not in df: 1
[datetime.date(2021, 10, 15)]


Data coverage is 99.9% complete, and it's the best time to move forward!